# ETL Pipeline — Superstore Dataset
### Phase 1: Extract, Transform, Load (Staging)
**Course:** Database Systems For Business  
**Institution:** FAST School of Management, NUCES

## Step 1 — Upload Source Files

In [ ]:
# Upload your source files when prompted
# Upload both: SuperStoreOrders.csv AND SuperStoreOrders.xlsx
from google.colab import files
uploaded = files.upload()

Saving SuperStoreOrders.csv to SuperStoreOrders.csv


## Step 2 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import unicodedata
import csv

## Step 3 — Extract
Read both CSV and Excel source files (satisfies the 2-format requirement).

In [ ]:
# =========================================
# EXTRACT — Read both file formats
# =========================================

# Read CSV
csv_df = pd.read_csv("SuperStoreOrders.csv")
print("CSV rows:", len(csv_df))




# Combine both into one dataframe
df = pd.concat([csv_df], ignore_index=True)
print("Combined rows before cleaning:", len(df))
print("Columns:", df.columns.tolist())

CSV rows: 51290
Combined rows before cleaning: 51290
Columns: ['order_id', 'order_date', 'ship_date', 'ship_mode', 'customer_name', 'segment', 'state', 'country', 'market', 'region', 'product_id', 'category', 'sub_category', 'product_name', 'sales', 'quantity', 'discount', 'profit', 'shipping_cost', 'order_priority', 'year']


## Step 4 — Transform
Clean, validate, and standardize the combined dataset.

In [ ]:
# =========================================
# TRANSFORM — Clean & Standardize
# =========================================

# --- Standardize column names ---
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

# --- Remove duplicates ---
before = len(df)
df.drop_duplicates(inplace=True)
print(f"Duplicates removed: {before - len(df)}")

# --- Numeric column cleaning ---
numeric_cols = ["sales", "quantity", "discount", "profit", "shipping_cost"]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# --- Fill missing values ---
df["sales"]         = df["sales"].fillna(df["sales"].mean())
df["quantity"]      = df["quantity"].fillna(1)
df["discount"]      = df["discount"].fillna(0)
df["profit"]        = df["profit"].fillna(df["profit"].mean())
df["shipping_cost"] = df["shipping_cost"].fillna(df["shipping_cost"].mean())
df["customer_name"] = df["customer_name"].fillna("Unknown")
df["product_name"]  = df["product_name"].fillna("Unknown")

# --- Date conversion (DD/MM/YYYY -> YYYY-MM-DD for MySQL) ---
df["order_date"] = pd.to_datetime(df["order_date"], format="%d/%m/%Y", errors="coerce").dt.strftime("%Y-%m-%d")
df["ship_date"]  = pd.to_datetime(df["ship_date"],  format="%d/%m/%Y", errors="coerce").dt.strftime("%Y-%m-%d")

# --- Trim whitespace from string columns ---
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].str.strip()

# --- Data Validation: check for invalid emails/formats if present ---
print("\nMissing values after cleaning:")
print(df.isnull().sum())

print("\nDataset shape after cleaning:", df.shape)

Duplicates removed: 0

Missing values after cleaning:
order_id          0
order_date        0
ship_date         0
ship_mode         0
customer_name     0
segment           0
state             0
country           0
market            0
region            0
product_id        0
category          0
sub_category      0
product_name      0
sales             0
quantity          0
discount          0
profit            0
shipping_cost     0
order_priority    0
year              0
dtype: int64

Dataset shape after cleaning: (51290, 21)


## Step 5 — Sample 5000 Rows

In [ ]:
# =========================================
# Take a sample of 5000 rows
# =========================================

df = df.sample(n=5000, random_state=42).reset_index(drop=True)
print("Final dataset shape:", df.shape)
print(df.head())

Final dataset shape: (5000, 21)
          order_id  order_date   ship_date       ship_mode    customer_name  \
0  IT-2014-4809306  2014-12-09  2014-12-14  Standard Class     Mike Kennedy   
1    ID-2014-46700  2014-10-03  2014-10-08  Standard Class  Penelope Sewall   
2   MX-2012-133732  2012-09-17  2012-09-19     First Class      Mark Packer   
3     TU-2014-5360  2014-06-30  2014-07-04  Standard Class    Liz Pelletier   
4    ID-2014-74763  2014-12-04  2014-12-09  Standard Class       Eva Jacobs   

       segment       state    country market          region  ...  \
0     Consumer   Stockholm     Sweden     EU           North  ...   
1  Home Office  Queensland  Australia   APAC         Oceania  ...   
2  Home Office     Chiapas     Mexico  LATAM           North  ...   
3     Consumer     Kayseri     Turkey   EMEA            EMEA  ...   
4     Consumer  Jawa Barat  Indonesia   APAC  Southeast Asia  ...   

          category sub_category                   product_name       sales  \


## Step 6 — Sanitize for MySQL Import
Remove characters that break the MySQL Workbench Import Wizard (commas inside fields, apostrophes, unicode).

In [ ]:
# =========================================
# SANITIZE — Make CSV safe for MySQL Wizard
# =========================================

def clean_for_mysql(val):
    if not isinstance(val, str):
        return val
    # Normalize unicode characters (e.g. é -> e, ô -> o)
    val = unicodedata.normalize("NFKD", val)
    val = val.encode("ascii", "ignore").decode("ascii")
    val = val.replace('"', '')      # remove double quotes
    val = val.replace("'", "")       # remove apostrophes
    val = val.replace(",", ";")       # commas -> semicolons
    val = val.replace("&", "and")     # ampersands -> and
    return val.strip()

for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].apply(clean_for_mysql)

print("Sanitization complete.")
print("Rows remaining:", len(df))

Sanitization complete.
Rows remaining: 5000


## Step 7 — Load (Export to CSV for MySQL Staging)
Export the cleaned dataset as a plain CSV ready for MySQL Workbench Table Data Import Wizard.

In [ ]:
# =========================================
# LOAD — Export cleaned CSV for MySQL import
# =========================================

output_filename = "cleaned_superstore_mysql.csv"

df.to_csv(
    output_filename,
    index=False,
    encoding="ascii",
    errors="replace"
)

print(f"Exported: {output_filename}")
print(f"Total rows: {len(df)}")
print(f"Total columns: {len(df.columns)}")

# Download the file
files.download(output_filename)

Exported: cleaned_superstore_mysql.csv
Total rows: 5000
Total columns: 21


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Step 8 — Also Export as Excel (bonus output)

In [ ]:
# =========================================
# Export cleaned dataset as Excel too
# =========================================

df.to_excel("cleaned_superstore.xlsx", index=False)
print("Excel file exported: cleaned_superstore.xlsx")

files.download("cleaned_superstore.xlsx")

Excel file exported: cleaned_superstore.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Summary
| Step | Action | Output |
|------|--------|--------|
| Extract | Read CSV + Excel | Combined DataFrame |
| Transform | Clean, deduplicate, fix dates, fill nulls | Clean DataFrame |
| Sanitize | Remove special chars for MySQL | MySQL-safe DataFrame |
| Load | Export CSV + Excel |  |

The exported CSV is ready to be imported into  in MySQL Workbench.